### facenet-pytorch LFW evaluation
This notebook demonstrates how to evaluate performance against the LFW dataset.

In [1]:
import sys
sys.path.append('C:\\Users\\Pol\\Documents\\code\\anonymisation')
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training, extract_face
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler, SequentialSampler
from torchvision import datasets, transforms
import numpy as np
import os

c:\Users\Pol\miniconda3\envs\xor\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [61]:
data_dir = 'C:\\Users\\Pol\\Documents\\code\\LFW'
# pairs_path = 'C:\\Users\\Pol\\Documents\\code\\LFW\\pairs.txt'
# pairs_path = 'C:\\Users\\Pol\\Documents\\code\\LFW\\matchpairsDevTrain.txt'
# pairs_path = 'C:\\Users\\Pol\\Documents\\code\\LFW\\mismatchpairsDevTrain.txt'

batch_size = 1
epochs = 15
workers = 0 if os.name == 'nt' else 8

In [18]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

Running on device: cuda:0


In [19]:
mtcnn = MTCNN(
    image_size=160,
    margin=14,
    device=device,
    selection_method='center_weighted_size',
    thresholds=[0.5, 0.7, 0.7]
)

In [62]:
# Define the data loader for the input set of images
orig_img_ds = datasets.ImageFolder(data_dir, transform=None)

In [63]:

paths_to_keep

[('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zumrati_Juma\\Zumrati_Juma_0001.jpg',
  'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zumrati_Juma\\Zumrati_Juma_0001.jpg'),
 ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg',
  'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg'),
 ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zydrunas_Ilgauskas\\Zydrunas_Ilgauskas_0001.jpg',
  'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zydrunas_Ilgauskas\\Zydrunas_Ilgauskas_0001.jpg'),
 ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zumrati_Juma\\Zumrati_Juma_0001.jpg',
  'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zumrati_Juma\\Zumrati_Juma_0001.jpg'),
 ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg',
  'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg'),
 ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zydrunas_I

In [64]:

# overwrites class labels in dataset with path so path can be used for saving output in mtcnn batches
orig_img_ds.samples = [
    (p, p)
    for p, _ in orig_img_ds.samples
]

#filter sample orig_img_dsz
paths_to_keep = []
for s in orig_img_ds.samples:
    for ks in ["Zumrati_Juma", "Zurab_Tsereteli", "Zydrunas_Ilgauskas"]:
        if ks in s[0]:
            paths_to_keep.append(s)
print(paths_to_keep)
orig_img_ds.samples = paths_to_keep

loader = DataLoader(
    orig_img_ds,
    num_workers=workers,
    batch_size=batch_size,
    collate_fn=training.collate_pil
)


[('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zumrati_Juma\\Zumrati_Juma_0001.jpg', 'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zumrati_Juma\\Zumrati_Juma_0001.jpg'), ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg', 'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg'), ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zydrunas_Ilgauskas\\Zydrunas_Ilgauskas_0001.jpg', 'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zydrunas_Ilgauskas\\Zydrunas_Ilgauskas_0001.jpg'), ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zumrati_Juma\\Zumrati_Juma_0001.jpg', 'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zumrati_Juma\\Zumrati_Juma_0001.jpg'), ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg', 'C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg'), ('C:\\Users\\Pol\\Documents\\code\\LFW\\imgs_cropped\\Zydrunas_Ilgauskas\\Zydru

In [21]:

# overwrites class labels in dataset with path so path can be used for saving output in mtcnn batches
orig_img_ds.samples = [
    (p, p)
    for p, _ in orig_img_ds.samples
]

loader = DataLoader(
    orig_img_ds,
    num_workers=workers,
    batch_size=batch_size,
    collate_fn=training.collate_pil
)


In [11]:
crop_paths = []
box_probs = []

for i, (x, b_paths) in enumerate(loader):
    print(x, b_paths)
    crops = [p.replace(data_dir, data_dir + '_cropped') for p in b_paths]
    tutu, batch_probs = mtcnn(x, save_path=crops, return_prob=True)
    crop_paths.extend(crops)
    print(tutu)
    print('\rBatch {} of {} of probs'.format(i + 1, len(loader), batch_probs), end='')


[<PIL.Image.Image image mode=RGB size=250x250 at 0x15D11DC91F0>] ['C:\\Users\\Pol\\Documents\\code\\LFW\\tut\\Marilyn_Monroe\\Marilyn_Monroe_0000.jpg']
[tensor([[[ 0.7305,  0.7305,  0.7305,  ...,  0.5195,  0.5117,  0.5039],
         [ 0.7227,  0.7305,  0.7305,  ...,  0.4883,  0.4883,  0.4961],
         [ 0.7227,  0.7227,  0.7227,  ...,  0.4570,  0.4648,  0.4727],
         ...,
         [-0.2070, -0.2148, -0.2227,  ..., -0.5195, -0.5117, -0.5117],
         [-0.2070, -0.2148, -0.2148,  ..., -0.5117, -0.5117, -0.5039],
         [-0.2148, -0.2227, -0.2227,  ..., -0.4961, -0.4961, -0.4883]],

        [[ 0.7305,  0.7305,  0.7305,  ...,  0.5195,  0.5117,  0.5039],
         [ 0.7227,  0.7305,  0.7305,  ...,  0.4883,  0.4883,  0.4961],
         [ 0.7227,  0.7227,  0.7227,  ...,  0.4570,  0.4648,  0.4727],
         ...,
         [-0.2070, -0.2148, -0.2227,  ..., -0.5195, -0.5117, -0.5117],
         [-0.2070, -0.2148, -0.2148,  ..., -0.5117, -0.5117, -0.5039],
         [-0.2148, -0.2227, -0.2227,

In [65]:
import cv2
import numpy as np
from facenet_pytorch import MTCNN
from PIL import Image

mtcnn = MTCNN(
    image_size=160,  # can be larger for detection
    margin=14,
    device=device,
    selection_method='center_weighted_size',
    thresholds=[0.5, 0.7, 0.7],
    post_process=False  # we will handle alignment
)

arcface_template = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041]
], dtype=np.float32)

for i, (x, b_paths) in enumerate(loader):
    print(f'\rProcessing batch {i + 1} of {len(loader)}', end='')
    print(x, b_paths)
    boxes, probs, landmarks = mtcnn.detect(x, landmarks=True)

    for pil_img, lm, save_path, prob in zip(x, landmarks, b_paths, probs):
        if lm is None:
            continue  # skip undetected faces

        # Convert PIL → NumPy array
        img_np = np.array(pil_img)  # shape (H, W, 3), RGB order

        print(lm)
        if lm.shape[0] > 1:  
            # lm: (batch, num_faces, 5, 2)
            print(probs)
            face_index = np.argmax(prob)  # pick best face
            lm = lm[face_index] 
        print(lm.shape)
        # Compute affine transform to ArcFace template
        M, _ = cv2.estimateAffinePartial2D(lm.astype(np.float32), arcface_template)
        aligned_face = cv2.warpAffine(img_np, M, (112, 112), borderValue=0)

        # Convert back to PIL if needed
        aligned_pil = Image.fromarray(aligned_face)

        # Save aligned image
        save_path = save_path.replace(data_dir, data_dir + '_aligned112')
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        aligned_pil.save(save_path)


Processing batch 1 of 6[<PIL.Image.Image image mode=RGB size=250x250 at 0x1D9C3B87220>] ['C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zumrati_Juma\\Zumrati_Juma_0001.jpg']
[[[106.54505920410156 108.50701904296875]
  [150.77114868164062 114.23206329345703]
  [135.06307983398438 125.56182861328125]
  [107.27819061279297 153.22738647460938]
  [145.28262329101562 156.14706420898438]]]
(1, 5, 2)
Processing batch 2 of 6[<PIL.Image.Image image mode=RGB size=250x250 at 0x1D9C3B87070>] ['C:\\Users\\Pol\\Documents\\code\\LFW\\imgs\\Zurab_Tsereteli\\Zurab_Tsereteli_0001.jpg']
[[[104.15863037109375 114.71038818359375]
  [145.4626007080078 112.56791687011719]
  [123.0294189453125 138.49606323242188]
  [108.80461120605469 157.6754150390625]
  [148.5331573486328 156.26473999023438]]

 [[210.14991760253906 147.137939453125]
  [233.07809448242188 144.875244140625]
  [229.88949584960938 158.656494140625]
  [215.46746826171875 172.87791442871094]
  [234.97024536132812 171.54638671875]]

 [[190.2281494140

In [33]:
import os

def get_subfolders(base_path):
    """Return a set of all subfolder paths relative to base_path."""
    subfolders = set()
    for root, dirs, _ in os.walk(base_path):
        for d in dirs:
            full_path = os.path.join(root, d)
            rel_path = os.path.relpath(full_path, base_path)
            subfolders.add(rel_path)
    return subfolders

def compare_folders(folder1, folder2):
    subfolders1 = get_subfolders(folder1)
    subfolders2 = get_subfolders(folder2)

    missing_in_folder2 = subfolders1 - subfolders2
    missing_in_folder1 = subfolders2 - subfolders1

    return missing_in_folder2, missing_in_folder1

if __name__ == "__main__":
    folder1 = "C:\\Users\\Pol\\Documents\\code\\LFW_aligned112\\imgs"
    folder2 = "C:\\Users\\Pol\\Documents\\code\\LFW\\imgs"

    missing_in_folder2, missing_in_folder1 = compare_folders(folder1, folder2)

    if missing_in_folder2:
        print(f"Subfolders in {folder1} but missing in {folder2}:")
        for sub in sorted(missing_in_folder2):
            print(f"  {sub}")
    else:
        print(f"No subfolders missing in {folder2}.")

    if missing_in_folder1:
        print(f"\nSubfolders in {folder2} but missing in {folder1}:")
        for sub in sorted(missing_in_folder1):
            print(f"  {sub}")
    else:
        print(f"\nNo subfolders missing in {folder1}.")


No subfolders missing in C:\Users\Pol\Documents\code\LFW\imgs.

Subfolders in C:\Users\Pol\Documents\code\LFW\imgs but missing in C:\Users\Pol\Documents\code\LFW_aligned112\imgs:
  Zumrati_Juma
  Zurab_Tsereteli
  Zydrunas_Ilgauskas


In [8]:
# Remove mtcnn to reduce GPU memory usage
del mtcnn
torch.cuda.empty_cache()

In [ ]:
def fixed_image_standardization(image_tensor):
    processed_tensor = (image_tensor - 127.5) / 128.0
    return processed_tensor

In [9]:
# create dataset and data loaders from cropped images output from MTCNN

trans = transforms.Compose([
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])

dataset = datasets.ImageFolder(data_dir + '_cropped', transform=trans)

embed_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SequentialSampler(dataset)
)

In [10]:
# Load pretrained resnet model
resnet = InceptionResnetV1(
    classify=False,
    pretrained='vggface2'
).to(device)

In [11]:
classes = []
embeddings = []
resnet.eval()
with torch.no_grad():
    for xb, yb in embed_loader:
        xb = xb.to(device)
        b_embeddings = resnet(xb)
        b_embeddings = b_embeddings.to('cpu').numpy()
        classes.extend(yb.numpy())
        embeddings.extend(b_embeddings)

In [12]:
embeddings_dict = dict(zip(crop_paths,embeddings))



#### Evaluate embeddings by using distance metrics to perform verification on the official LFW test set.

The functions in the next block are copy pasted from `facenet.src.lfw`. Unfortunately that module has an absolute import from `facenet`, so can't be imported from the submodule

added functionality to return false positive and false negatives

In [13]:
from sklearn.model_selection import KFold
from scipy import interpolate

# LFW functions taken from David Sandberg's FaceNet implementation
def distance(embeddings1, embeddings2, distance_metric=0):
    if distance_metric==0:
        # Euclidian distance
        diff = np.subtract(embeddings1, embeddings2)
        dist = np.sum(np.square(diff),1)
    elif distance_metric==1:
        # Distance based on cosine similarity
        dot = np.sum(np.multiply(embeddings1, embeddings2), axis=1)
        norm = np.linalg.norm(embeddings1, axis=1) * np.linalg.norm(embeddings2, axis=1)
        similarity = dot / norm
        dist = np.arccos(similarity) / math.pi
    else:
        raise 'Undefined distance metric %d' % distance_metric

    return dist

def calculate_roc(thresholds, embeddings1, embeddings2, actual_issame, nrof_folds=10, distance_metric=0, subtract_mean=False):
    assert(embeddings1.shape[0] == embeddings2.shape[0])
    assert(embeddings1.shape[1] == embeddings2.shape[1])
    nrof_pairs = min(len(actual_issame), embeddings1.shape[0])
    nrof_thresholds = len(thresholds)
    k_fold = KFold(n_splits=nrof_folds, shuffle=False)

    tprs = np.zeros((nrof_folds,nrof_thresholds))
    fprs = np.zeros((nrof_folds,nrof_thresholds))
    accuracy = np.zeros((nrof_folds))

    is_false_positive = []
    is_false_negative = []

    indices = np.arange(nrof_pairs)

    for fold_idx, (train_set, test_set) in enumerate(k_fold.split(indices)):
        if subtract_mean:
            mean = np.mean(np.concatenate([embeddings1[train_set], embeddings2[train_set]]), axis=0)
        else:
          mean = 0.0
        dist = distance(embeddings1-mean, embeddings2-mean, distance_metric)

        # Find the best threshold for the fold
        acc_train = np.zeros((nrof_thresholds))
        for threshold_idx, threshold in enumerate(thresholds):
            _, _, acc_train[threshold_idx], _ ,_ = calculate_accuracy(threshold, dist[train_set], actual_issame[train_set])
        best_threshold_index = np.argmax(acc_train)
        for threshold_idx, threshold in enumerate(thresholds):
            tprs[fold_idx,threshold_idx], fprs[fold_idx,threshold_idx], _, _, _ = calculate_accuracy(threshold, dist[test_set], actual_issame[test_set])
        _, _, accuracy[fold_idx], is_fp, is_fn = calculate_accuracy(thresholds[best_threshold_index], dist[test_set], actual_issame[test_set])

        tpr = np.mean(tprs,0)
        fpr = np.mean(fprs,0)
        is_false_positive.extend(is_fp)
        is_false_negative.extend(is_fn)

    return tpr, fpr, accuracy, is_false_positive, is_false_negative

def calculate_accuracy(threshold, dist, actual_issame):
    predict_issame = np.less(dist, threshold)
    tp = np.sum(np.logical_and(predict_issame, actual_issame))
    fp = np.sum(np.logical_and(predict_issame, np.logical_not(actual_issame)))
    tn = np.sum(np.logical_and(np.logical_not(predict_issame), np.logical_not(actual_issame)))
    fn = np.sum(np.logical_and(np.logical_not(predict_issame), actual_issame))

    is_fp = np.logical_and(predict_issame, np.logical_not(actual_issame))
    is_fn = np.logical_and(np.logical_not(predict_issame), actual_issame)

    tpr = 0 if (tp+fn==0) else float(tp) / float(tp+fn)
    fpr = 0 if (fp+tn==0) else float(fp) / float(fp+tn)
    acc = float(tp+tn)/dist.size
    return tpr, fpr, acc, is_fp, is_fn

def calculate_val(thresholds, embeddings1, embeddings2, actual_issame, far_target, nrof_folds=10, distance_metric=0, subtract_mean=False):
    assert(embeddings1.shape[0] == embeddings2.shape[0])
    assert(embeddings1.shape[1] == embeddings2.shape[1])
    nrof_pairs = min(len(actual_issame), embeddings1.shape[0])
    nrof_thresholds = len(thresholds)
    k_fold = KFold(n_splits=nrof_folds, shuffle=False)

    val = np.zeros(nrof_folds)
    far = np.zeros(nrof_folds)

    indices = np.arange(nrof_pairs)

    for fold_idx, (train_set, test_set) in enumerate(k_fold.split(indices)):
        if subtract_mean:
            mean = np.mean(np.concatenate([embeddings1[train_set], embeddings2[train_set]]), axis=0)
        else:
          mean = 0.0
        dist = distance(embeddings1-mean, embeddings2-mean, distance_metric)

        # Find the threshold that gives FAR = far_target
        far_train = np.zeros(nrof_thresholds)
        for threshold_idx, threshold in enumerate(thresholds):
            _, far_train[threshold_idx] = calculate_val_far(threshold, dist[train_set], actual_issame[train_set])
        if np.max(far_train)>=far_target:
            f = interpolate.interp1d(far_train, thresholds, kind='slinear')
            threshold = f(far_target)
        else:
            threshold = 0.0

        val[fold_idx], far[fold_idx] = calculate_val_far(threshold, dist[test_set], actual_issame[test_set])

    val_mean = np.mean(val)
    far_mean = np.mean(far)
    val_std = np.std(val)
    return val_mean, val_std, far_mean

def calculate_val_far(threshold, dist, actual_issame):
    predict_issame = np.less(dist, threshold)
    true_accept = np.sum(np.logical_and(predict_issame, actual_issame))
    false_accept = np.sum(np.logical_and(predict_issame, np.logical_not(actual_issame)))
    n_same = np.sum(actual_issame)
    n_diff = np.sum(np.logical_not(actual_issame))
    val = float(true_accept) / float(n_same)
    far = float(false_accept) / float(n_diff)
    return val, far



def evaluate(embeddings, actual_issame, nrof_folds=10, distance_metric=0, subtract_mean=False):
    # Calculate evaluation metrics
    thresholds = np.arange(0, 4, 0.01)
    embeddings1 = embeddings[0::2]
    embeddings2 = embeddings[1::2]
    tpr, fpr, accuracy, fp, fn  = calculate_roc(thresholds, embeddings1, embeddings2,
        np.asarray(actual_issame), nrof_folds=nrof_folds, distance_metric=distance_metric, subtract_mean=subtract_mean)
    thresholds = np.arange(0, 4, 0.001)
    val, val_std, far = calculate_val(thresholds, embeddings1, embeddings2,
        np.asarray(actual_issame), 1e-3, nrof_folds=nrof_folds, distance_metric=distance_metric, subtract_mean=subtract_mean)
    return tpr, fpr, accuracy, val, val_std, far, fp, fn

def add_extension(path):
    if os.path.exists(path+'.jpg'):
        return path+'.jpg'
    elif os.path.exists(path+'.png'):
        return path+'.png'
    else:
        raise RuntimeError('No file "%s" with extension png or jpg.' % path)

def get_paths(lfw_dir, pairs):
    nrof_skipped_pairs = 0
    path_list = []
    issame_list = []
    for pair in pairs:
        if len(pair) == 3:
            path0 = add_extension(os.path.join(lfw_dir, pair[0], pair[0] + '_' + '%04d' % int(pair[1])))
            path1 = add_extension(os.path.join(lfw_dir, pair[0], pair[0] + '_' + '%04d' % int(pair[2])))
            issame = True
        elif len(pair) == 4:
            path0 = add_extension(os.path.join(lfw_dir, pair[0], pair[0] + '_' + '%04d' % int(pair[1])))
            path1 = add_extension(os.path.join(lfw_dir, pair[2], pair[2] + '_' + '%04d' % int(pair[3])))
            issame = False
        if os.path.exists(path0) and os.path.exists(path1):    # Only add the pair if both paths exist
            path_list += (path0,path1)
            issame_list.append(issame)
        else:
            nrof_skipped_pairs += 1
    if nrof_skipped_pairs>0:
        print('Skipped %d image pairs' % nrof_skipped_pairs)

    return path_list, issame_list

def read_pairs(pairs_filename):
    pairs = []
    with open(pairs_filename, 'r') as f:
        for line in f.readlines()[1:]:
            pair = line.strip().split()
            pairs.append(pair)
    return np.array(pairs, dtype=object)

In [14]:
pairs = read_pairs(pairs_path)
path_list, issame_list = get_paths(data_dir+'_cropped', pairs)
embeddings = np.array([embeddings_dict[path] for path in path_list])

tpr, fpr, accuracy, val, val_std, far, fp, fn = evaluate(embeddings, issame_list)

In [15]:
print(accuracy)
np.mean(accuracy)



[0.995      0.995      0.99166667 0.99       0.99       0.99666667
 0.99       0.995      0.99666667 0.99666667]


0.9936666666666666